# 🏢 Telecom Customer Churn  End-to-End BI Pipeline
**Author:** Haseeb Waqas  
**Stack:** Python (ETL) → SQLite (SQL) → Power BI (Dashboard)  
**Dataset:** IBM Telco Customer Churn (7,043 records)

---

## Pipeline Architecture
```
┌─────────────────────────────────────────────────────────────┐
│                    BI PIPELINE OVERVIEW                      │
│                                                              │
│  [Excel File]  [JSON API]                                    │
│       ↓              ↓                                       │
│    ┌──────────────────────┐                                  │
│    │   ETL Pipeline       │  ← Step 2 (replaces SSIS)        │
│    │  Extract→Transform   │                                  │
│    │  →Load to SQLite     │                                  │
│    └──────────┬───────────┘                                  │
│               ↓                                              │
│    ┌──────────────────────┐                                  │
│    │   SQL Analysis       │  ← Step 3                        │
│    │  Clean + Aggregate   │                                  │
│    └──────────┬───────────┘                                  │
│               ↓                                              │
│    ┌──────────────────────┐                                  │
│    │   Power BI Dashboard │  ← Step 4                        │
│    │  KPIs + Charts       │                                  │
│    └──────────────────────┘                                  │
└─────────────────────────────────────────────────────────────┘
```


In [9]:
import warnings
warnings.filterwarnings('ignore')

## Step 1  Data Sourcing

In [2]:
%run ../pipeline/step1_data_sourcing.py

✅ Base dataset loaded: 7,043 rows
✅ Source A saved: source_a_demographics.xlsx  (7,055 rows, includes duplicates & nulls)
✅ Source B saved: source_b_billing_api.json   (7,043 records, includes data quality issues)

── Source Summary ──
   Source A (Excel) : 7,055 rows | 8 columns | demographics
   Source B (API)   : 7,043 rows | 11 columns | billing

✅ STEP 1 COMPLETE — Raw sources ready for ETL pipeline
   → Run step2_etl_pipeline.py next


## Step 2 ETL Pipeline

In [10]:
%run ../pipeline/step2_etl_pipeline.py

ETL PIPELINE — Telecom Customer Churn
Started : 2026-04-30 05:22:56

── EXTRACT ──
   Source A extracted : 7,055 rows from Excel
   Source B extracted : 7,043 rows from JSON API

── TRANSFORM — Demographics ──
   Duplicates removed  : 12 rows dropped → 7,043 remain
   Gender standardized : all values → 'Male' / 'Female'
   Partner nulls filled: 149 missing → filled with 'No'
   SeniorCitizen       : ✅ All values valid (0/1)

── TRANSFORM — Billing ──
   TotalCharges fixed  : 90 blanks → estimated from MonthlyCharges
   MonthlyCharges      : 31 negative values removed
   Churn encoded       : 'Yes'→1, 'No'→0

── TRANSFORM — Merge & Feature Engineering ──
   Merge result        : 7,012 rows (inner join on customerID)
   Features created    : Tenure_Group, Charge_Tier, Revenue_Segment, Is_Senior

── LOAD → SQLite Database ──
   ✅ fact_customers    : 7,012 rows loaded
   ✅ dim_contract      : 3 contract types loaded
   ✅ agg_churn_summary : 572 aggregated rows loaded

── Database Validatio

## Step 3  SQL Analysis

In [4]:
%run ../pipeline/step3_sql_analysis.py

SQL DATA CLEANING & ANALYSIS — Telecom Churn

── SQL Data Quality Checks ──

📋 NULL CHECK:
 null_customerID  null_MonthlyCharges  null_Churn  null_Contract  total_rows
               0                    0           0              0        7012

📋 MONTHLY CHARGES RANGE:
 min_charge  max_charge  avg_charge  negative_charges
       18.0       120.0       65.01                 0

📋 CHURN VALUES:
Churn  count
   No   4087
  Yes   2925

── SQL Business Analysis ──

📊 KPI SUMMARY:
 Total_Customers  Total_Churned  Churn_Rate_Pct  Avg_Monthly_Charge  Avg_Tenure_Months  Total_Revenue
            7012           2925           41.71               65.01               26.6     11441660.0

📊 CHURN BY CONTRACT:
      Contract  Total_Customers  Churned  Churn_Rate_Pct  Avg_Monthly_Charge
Month-to-month             3919     2088           53.28               64.99
      One year             1690      504           29.82               64.68
      Two year             1403      333           23.73       

## Step 4  Verify Output Files

In [6]:
import pandas as pd
import sqlite3
import os

print('── Output Files ──')
files = [
    '../data/raw/source_a_demographics.xlsx',
    '../data/raw/source_b_billing_api.json',
    '../data/telecom_churn.db',
    '../data/telco_master_powerbi.csv',
    '../data/sql_analysis_results.xlsx',
]
for f in files:
    exists = 'T' if os.path.exists(f) else 'F'
    print(f'   {exists} {f}')

print('\n── Database Tables ──')
conn = sqlite3.connect('../data/telecom_churn.db')
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
for t in tables['name']:
    n = pd.read_sql(f'SELECT COUNT(*) as n FROM {t}', conn)['n'][0]
    print(f'    {t}: {n:,} rows')
conn.close()

print('\n── Power BI Input File ──')
df = pd.read_csv('../data/telco_master_powerbi.csv')
print(f'   Rows    : {len(df):,}')
print(f'   Columns : {df.shape[1]}')
print(f'   Columns : {list(df.columns)}')

── Output Files ──
   T ../data/raw/source_a_demographics.xlsx
   T ../data/raw/source_b_billing_api.json
   T ../data/telecom_churn.db
   T ../data/telco_master_powerbi.csv
   T ../data/sql_analysis_results.xlsx

── Database Tables ──
    fact_customers: 7,012 rows
    dim_contract: 3 rows
    agg_churn_summary: 572 rows

── Power BI Input File ──
   Rows    : 7,012
   Columns : 23
   Columns : ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'TechSupport', 'StreamingTV', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'Churn_Binary', 'Tenure_Group', 'Charge_Tier', 'Revenue_Segment', 'Is_Senior']


## Step 5  Key Business Findings

In [11]:
conn = sqlite3.connect('../data/telecom_churn.db')

kpis = pd.read_sql("""
    SELECT
        COUNT(*)                          AS Total_Customers,
        SUM(Churn_Binary)                 AS Churned,
        ROUND(AVG(Churn_Binary)*100,2)    AS Churn_Rate_Pct,
        ROUND(AVG(MonthlyCharges),2)      AS Avg_Monthly_Charge,
        ROUND(SUM(TotalCharges),0)        AS Total_Revenue,
        ROUND(SUM(CASE WHEN Churn_Binary=1
              THEN MonthlyCharges ELSE 0 END),0) AS Monthly_Revenue_At_Risk
    FROM fact_customers
""", conn)

print('='*55)
print('EXECUTIVE SUMMARY')
print('='*55)
for col in kpis.columns:
    val = kpis[col][0]
    if 'Revenue' in col or 'Charge' in col:
        print(f'   {col:<30}: ${val:,.0f}')
    elif 'Pct' in col or 'Rate' in col:
        print(f'   {col:<30}: {val}%')
    else:
        print(f'   {col:<30}: {val:,}')

conn.close()

print('\n BUSINESS RECOMMENDATIONS')
recs = [
    'Month-to-month customers churn at 53%  offer loyalty discounts at 3-month mark',
    'New customers (0-12 months) churn at 51%  implement onboarding check-in program',
    'Fiber optic users churn most despite premium pricing  review service quality',
    'Electronic check users have highest churn  incentivize auto-pay enrollment',
]
for i, r in enumerate(recs, 1):
    print(f'   {i}. {r}')

EXECUTIVE SUMMARY
   Total_Customers               : 7,012
   Churned                       : 2,925
   Churn_Rate_Pct                : 41.71%
   Avg_Monthly_Charge            : $65
   Total_Revenue                 : $11,441,660
   Monthly_Revenue_At_Risk       : $195,501

 BUSINESS RECOMMENDATIONS
   1. Month-to-month customers churn at 53%  offer loyalty discounts at 3-month mark
   2. New customers (0-12 months) churn at 51%  implement onboarding check-in program
   3. Fiber optic users churn most despite premium pricing  review service quality
   4. Electronic check users have highest churn  incentivize auto-pay enrollment
